# PRÁCTICA 2 SEMANA 1

**Se recomienda ejecutar los códigos de este jupyer notebook en la misma carpeta donde reside dicho documento, debido a la dependencia de ciertos documentos para el correcto uso del mismo.**

En primer lugar, vamos a importar las librerías necesarias y vamos a importar los csv necesarios. Defino los dataframes dos veces, esto está explicado más adelante.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as snsz
from sklearn.linear_model import LinearRegression
import scipy.stats as stats
from sklearn.metrics import r2_score


In [ ]:
housing_test = pd.read_csv("dataset_housing_preprocesado_test.csv")
housing_train = pd.read_csv("dataset_housing_preprocesado_train.csv")
test = housing_test.drop(columns=["SalePrice"])
train = housing_train.drop(columns=["SalePrice"])

En esta semana 1 nos han proporcionado dos datasets, uno es el que entrena el modelo(train) y el otro es con el que comprueba si el modelo funciona correctamente(test), por ello tendremos que tratarlos de maneras diferentes.

In [ ]:
test

In [ ]:
train

Ahora, vamos a seguir los diferentes pasos para conseguir el mejor resultado.
#### PASO 1: Eliminación de variables colineales.
Si las variables de entrada al modelo son linealmente dependientes, la matriz de covarianza no es invertible. Como consecuencia, el problema de regresión por mínimos cuadrados no está bien definido.

Para abordar esto, vamos a hallar la matriz de covarianza y su rango para ver cuales son dependientes, buscamos hallar las variables dependientes para eliminarlas y definir correctamente el problema de regresión por mínimos de cuadrados, en este caso vamos a centrarnos para eliminar las variables dependientes de train, dado que nuestro modelo va a entrenar con todas las variables que no eliminemos el test tendrá que hacerlo también con todas las variables que no hemos eliminado del train. Es decir, vamos a eliminar las variables dependientes o colineales del train y esas mismas variables vamos a eliminarlas en el test.

Pero antes de eso, vamos a redefinir los datasets. Esto es debido a que el atributo a predecir, que en este caso podría ser el precio de la vivienda, no es una variable explicita como podría ser una fecha o un id sino la variable dependiente por defecto. Utilizar las matrices de correlación y de covarianza ha de servirnos únicamente para indicar la independencia entre las variables de los dataframes.

In [ ]:
test_cov = test.cov()
train_cov = train.cov()
test_corr = test.corr()
train_corr = train.corr()

In [ ]:
test_cov

In [ ]:
train_cov

In [ ]:
test_range = np.linalg.matrix_rank(test_cov)
train_range = np.linalg.matrix_rank(train_cov)
print(f"Test:{test_cov.shape}, Train {train_cov.shape}")
print(f"Test {test_range}, Train {train_range}")

Vemos que tanto Train como Test tienen el mismo número de variables: 153 (contando con la variable a predecir). Sin embargo, a la hora de ajustar las variables usando los rangos de las matrices de covarianzas vemos que el dataset Train tiene 12 variables dependientes y el dataset Test tiene 21 variables dependientes. Vamos a hallar cuales son estas 12 variables y a eliminarlas de ambos dataframes utilizando la matriz de correlación. 

In [ ]:
# Seguimos el pseudocódigo del pdf:
# 1,2. Definimos cols y calculamos rank
cols = list(train.columns)
rank = np.linalg.matrix_rank(train[cols].cov())
# 3 Generamos un bucle while para que lo haga para cada columna
i = 0
while i < len(cols):
    col = cols[i]
    # Creamos una copia de las columnas sin la actual
    cols2 = cols.copy()
    cols2.remove(col)
    # Calculamos el rango sin esa columna
    train2 = train[cols2]
    rank2 = np.linalg.matrix_rank(train2.cov())
    # Si el rango no cambia, eliminamos definitivamente la columna
    if rank == rank2:
        cols = cols2
        i = 0
    else:
        i += 1
        
# Ahora eliminamos las variables dependientes redefiniendo el dataset
train = train[cols]
test = test[cols]
# Comprobamos
print(train.shape)
print(test.shape)
# Comprobamos el rango otra vez.
test_cov = test.cov()
train_cov = train.cov()
test_range = np.linalg.matrix_rank(test_cov)
train_range = np.linalg.matrix_rank(train_cov)
print(f"Test:{test_cov.shape}, Train {train_cov.shape}")
print(f"Test {test_range}, Train {train_range}")

Ahora tenemos nuestros datasets sin nuestra variable a predecir y sin variables colineales. Tenemos un dataset de entrenamiento con solo variables independientes. Por tanto, podemos pasar al Paso 2 de la práctica.

#### PASO 2: Construcción de un primer modelo
Primero, vamos a construir un modelo de regresión lineal usando sklearn y vamos a calcular su coeficiente de determinación R2 en ambos datasets. Esto lo realizamos para evaluar si la reducción de variables ha mantenido la capacidad predictiva del modelo. Un R2 similar al original indica que las variables eliminadas eran redundantes, mientras que un valor muy diferente puede sugerir pérdida de información relevante.

In [ ]:
# Vamos a definir el modelo lineal utilizando ambos datasets.
# Utilizamos test y train para definir el eje X porque no tiene el atributo "SalePrice",
# que es el atributo a predecir o eso creemos y vamos a entrenar el modelo.
X_test = test
Y_test = housing_test["SalePrice"]
X_train = train
Y_train = housing_train["SalePrice"]
print(X_test.shape,Y_test.shape,X_train.shape,Y_train.shape)
modelo = LinearRegression()
modelo.fit(X_train,Y_train)

In [ ]:
# Hallamos el coeficiente R2 de test y train
print(modelo.score(X_train,Y_train))
print(modelo.score(X_test,Y_test))

El modelo de regresión lineal alcanza un coeficiente de determinación R2 = 0.94 en el conjunto de entrenamiento y R2 = 0.88 en el conjunto de test.

Dado que ambos valores son elevados y la diferencia entre ellos es moderada, no hay sobreajuste. El modelo logra un equilibrio razonable de  ajuste. Ahora vamos a ver si nuestro modelo consigue un precio estimado con el modelo de training y vamos a visualizarlo creando un scatter plot.

In [ ]:
# Vamos a definir las predicciones y visualizarlas mediante un scatter plot
train_predict = modelo.predict(X_train)
test_predict = modelo.predict(X_test)
min_test = min(Y_test) 
max_test = max(Y_test)
min_train = min (Y_train)
max_train = max(Y_train)
print(min_test,min_train,max_test,max_train)

plt.scatter(Y_test, test_predict, alpha=0.5)
plt.plot([min_test,max_test], [min_test,max_test], "r")
plt.xlabel("Precio real con test")
plt.ylabel("Precio estimado con test")
plt.title("Test: Precio real vs estimado")
plt.grid()
plt.show()

plt.scatter(Y_train, train_predict, alpha=0.5)
plt.plot([min_train,max_train], [min_train,max_train], "r")
plt.xlabel("Precio real con train")
plt.ylabel("Precio estimado con train")
plt.title("Train: Precio real vs estimado")
plt.grid()
plt.show()

Con los scatter plots, quitando de lado los outliers, podemos ver que los puntos o datos no se alejan de la diagonal. Al agruparse junto a la diagonal podemos concluir que el modelo ajusta bien. Ahora vamos a hallar los coeficientes del modelo lineal ajustado y su termino independiente. Con estos coeficientes vamos a calcular las predicciones nuevamente y vamos a ver si coinciden con las de modelo.predict().

In [ ]:
print(modelo.coef_)
print(modelo.intercept_)

Ahora, para calcular manualmente la predicción tenemos que seguir $\hat{y} = X \cdot \beta + \beta_0$. Qué es la fórmula que utiliza sklearn para hacer el modelo predictivo. Donde $\beta_0$ es el término independiente y todas las demás betas son los coeficientes. 

In [ ]:
# y = X * ß + ß_0. Utilizamos np.dot para calcular la multiplicación con cada coeficiente
y_pred_manual = np.dot(X_test, modelo.coef_) + modelo.intercept_

# Comprobamos igualdad con np.allclose que comprueba 
# si todos los valores son iguales permitiendo pequeñas diferenia numéricas.
np.allclose(test_predict, y_pred_manual)

Vemos que el modelo predictivo coincide con el modelo manual que hemos creado. Ahora vamos a ver si los residuos siguen una distribución gaussiana con un histograma y un qq-plot.

In [ ]:
# Obtenemos los residuos (Valor real - valor predictivo)
residuos_train = Y_train - modelo.predict(X_train)
residuos_test = Y_test - modelo.predict(X_test)

# Histograma y qq-plot TRAIN
plt.hist(residuos_train, bins=30, alpha=0.7)
plt.title("Distribución de residuos con Train")
plt.xlabel("Residuo")
plt.ylabel("Frecuencia de la aparición")
plt.grid()
plt.show()

stats.probplot(residuos_train, dist="norm", plot=plt)
plt.title("QQ-plot de residuos con Train")
plt.show()

# Histograma y qq-plot TEST
plt.hist(residuos_test, bins=30, alpha=0.7)
plt.title("Distribución de residuos Test")
plt.xlabel("Residuo")
plt.ylabel("Frecuencia de la aparición")
plt.grid()
plt.show()

stats.probplot(residuos_test, dist="norm", plot=plt)
plt.title("QQ-plot de residuos con Test")
plt.show()

Para que siga una distribución gaussiana tiene que darse en los histogramas una especie de relación a una campana y en cuanto a los qq-plots la mayoría de los valores se han de solapar entorno a la diagonal. Como podemos ver todo lo mencionado se cumple, así que por tanto si que sigue una distribución gaussiana tanto en test como en train.

Para seguir vamos a explorar el efecto de transformar de manera no lineal 
el target usando la transformación logarítmica, utilizando un histograma del target ("SalePrice") sin transformar y comparándolo con el transformado.

In [ ]:
# Histograma con el target original.
plt.hist(Y_train, bins = 30, alpha = 0.7)
plt.title("Distribución con el target original")
plt.grid()
plt.show()
# Histograma con el target transformado.
plt.hist(np.log(Y_train), bins = 30, alpha = 0.7)
plt.title("Distribución con el target transformado")
plt.grid()
plt.show()

Fijándonos en el histograma del target sin modificar vemos que la distribución recae en la parte de la izquierda, teniendo más sentido ya que suele haber más casas baratas que caras. Por otra parte, fijándonos en el modelo logarítmico el histograma sigue mucho más de cerca una forma de distribución normal.

Por último, vamos a volver a crear un modelo lineal, esta vez vamos a utilizar la versión logarítmica con el atributo target transformado. Para ello tendremos que volver a convertir las predicciones a dólares y volviendo a hallar el R2

In [ ]:
# Creamos el modelo lineal con train.
modelo_log = LinearRegression()
modelo_log.fit(X_train, np.log(Y_train))
# Creamos las predicciones y ajustamos el target.
prediction_train_log = np.exp(modelo_log.predict(X_train))
prediction_test_log = np.exp(modelo_log.predict(X_test))
#Calculamos R2 con r2_score
r2_train_log = r2_score (Y_train,prediction_train_log)
r2_test_log =r2_score(Y_test, prediction_test_log)

print(r2_test_log,r2_train_log)

**Los valores R2 de nuestro primer modelo eran:**

Train: 0.9424614967584221

Test: 0.8768548653213977

**Los valores R2 de nuestro modelo transformado son:**

Train:0.9706299000102838

Test: 0.9296046419335727

Dado que los datos de R2 de nuestro modelo logarítmico están más cerca de 1, quiere decir que son mucho mejores que los de nuestro modelo original. Por tanto y en conclusión, si tuvieramos que elegir un modelo para predecir el precio de venta de una vivienda utilizaría el modelo transformado, ya que se acerca más a la realidad.

# PRÁCTICA 2 SEMANA 2

#### Paso 1: Construcción de un modelo lineal con librería statsmodels
Ahora, hemos de repetir el mismo proceso de la semana 1 pero utilizando otra librería statsmodels la cual vamos a importar a continuación. Para construir este modelo lineal vamos a utilizar diferentes herramientas y hallar ciertas estadísticas para la comprobación de la eficacia y eficiencia del modelo.

In [ ]:
import statsmodels.api as sm
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)


# Definir el modelo y lo entrenamos
modelo_sm = sm.OLS(Y_train, X_train_sm)
modelo_ajustado = modelo_sm.fit()
predicciones_test = modelo_ajustado.predict(X_test_sm)

coef_sm = modelo_ajustado.params.drop('const').values
coef_sklearn = modelo.coef_

print(modelo_ajustado.params.values)
print("")
print(modelo_ajustado.rsquared)
print(r2_score(Y_test, predicciones_test))
print(np.allclose(coef_sm, coef_sklearn))

Tenemos que:

- Sklearn:
  
0.9424614967584221

0.8768548653213977

- StatsModels:

0.9424614967584221

0.8768548653212765

Comparando los valores no hay diferencia entre los modelos, tampoco vemos diferencias entre los coeficientes ni tampoco en los coeficientes de determinación. También vamos a ver el QQ-plot.

In [ ]:
# Hallamos los residuos
residuos = modelo_ajustado.resid

# QQ-plot
stats.probplot(residuos, dist="norm", plot=plt)
plt.title("QQ-plot de los residuos (StatsModels)")
plt.xlabel("Cuantiles teóricos (Normal)")
plt.ylabel("Cuantiles muestrales (Residuos)")
plt.grid()
plt.show()

In [ ]:
# Información estadística del modelo 
print(modelo_ajustado.summary())

In [ ]:
# Información detallada
std_errors = modelo_ajustado.bse
# p-values
p_values = modelo_ajustado.pvalues
# Intervalo confianza (95%)
conf_intervals = modelo_ajustado.conf_int(alpha=0.05)

summary_df = pd.DataFrame({
    'coef': modelo_ajustado.params,
    'std_error': std_errors,
    'p_value': p_values,
    'CI_lower': conf_intervals[0],
    'CI_upper': conf_intervals[1]
})

print(summary_df)


Entonces las variables con p-valores mayores a 0.05 y cuyos intervalos de confianza incluyen el cero no aportan información estadísticamente significativa al modelo.
En este caso, destacan como no relevantes variables como LotFrontage, LotShape, LandContour, YearRemodAdd, GarageArea, Fence y varias dummies de MSSubClass, MSZoning y Neighborhood.

# SEMANA 3
